[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/math_module2/blob/main/ch9_newton_gradient_descent.ipynb)

# Chapter 9 — Companion Notebook
## When You Can't Solve It, Chase It
*How to find an answer by taking smarter and smarter guesses*

The centerpiece lab of the book, and where most of Chapter 9's practice lives. Five live experiments turn iteration into something you *watch*: a gradient-descent playground where one dial — the learning rate — slides the glide into a spiral into an explosion; Newton's method doubling its correct digits on $\sqrt2$ and then flying apart on $x^{1/3}$; a two-valley landscape where the *starting point* alone decides which minimum you fall into; a head-to-head step count of Newton against gradient descent; and your first real optimizer — momentum — on a steep, ill-conditioned bowl.

> **How to use this notebook.** Run every cell from the top (Shift+Enter). Each widget below is live — drag the sliders and watch the picture respond. Requires `ipywidgets` and `matplotlib`.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from ipywidgets import (interact, interactive, fixed, Dropdown, Checkbox,
                        Button, Output, VBox, HBox,
                        FloatSlider, IntSlider, FloatLogSlider)
from IPython.display import HTML, display, Markdown

# Book 2 palette — matches the printed figures in figures/png/
BLUE="#1f4e79"; RED="#c0392b"; GREEN="#2e8b57"; ORANGE="#e08e0b"
PURPLE="#6c3483"; TEAL="#1b9e9e"; GRAY="#7f8c8d"; DARK="#2c3e50"
SHADE="#9fc5e8"; SHADE2="#f4b6ad"; SHADEG="#a9dfbf"; LIGHT="#d6dbdf"

plt.rcParams.update({
    "figure.figsize": (7.2, 4.5), "figure.dpi": 96,
    "font.family": "serif", "mathtext.fontset": "cm", "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": LIGHT, "grid.linewidth": 0.7,
    "lines.linewidth": 2.2, "lines.solid_capstyle": "round",
})

def axes(ax=None, title=None):
    '''A clean axes with a light grid; returns the axes.'''
    if ax is None:
        _, ax = plt.subplots()
    if title:
        ax.set_title(title, color=DARK)
    return ax


## 1. The gradient-descent playground — turn the dial, watch the regime change

Here is the whole chapter in one picture. We minimize the simplest valley there is, the bowl $f(x)=x^2$, whose bottom is plainly at $x=0$. Starting from a guess $x_0$, gradient descent repeats one line:

$$x_{k+1}=x_k-\eta\,f'(x_k)=x_k-\eta\,(2x_k)=(1-2\eta)\,x_k.$$

Every step just multiplies your position by the fixed number $1-2\eta$ — so the whole fate of the run is decided by the single quantity $|1-2\eta|$, exactly the geometric-sequence test from Chapter 1. Drag the **start** $x_0$ (this is your 'draggable start') and, more importantly, the **learning rate** $\eta$, and watch the path on the bowl pass through four regimes:

- **smooth glide** ($\eta$ small, factor in $(0,1)$) — creeps patiently down one side and lands;
- **shrinking spiral** ($\tfrac12<\eta<1$, factor in $(-1,0)$) — overshoots the bottom each step, but the overshoots shrink, so it zig-zags in;
- **stuck bounce** ($\eta=1$, factor $=-1$) — ping-pongs between $\pm x_0$ forever, never arriving;
- **explosion** ($\eta>1$, factor $<-1$) — each step lands farther out than the last, and it detonates.

The magic value $\eta=\tfrac12$ makes the factor $0$: gradient descent jumps to the exact minimum in **one step**. Try it.

In [2]:
def _playground(x0=1.0, eta=0.1, n_steps=12):
    f  = lambda x: x**2
    fp = lambda x: 2.0 * x
    # Run the descent x_{k+1} = x_k - eta*f'(x_k), guarding overflow.
    xs = [float(x0)]
    for _ in range(n_steps):
        nxt = xs[-1] - eta * fp(xs[-1])
        if not np.isfinite(nxt) or abs(nxt) > 1e6:
            xs.append(nxt); break
        xs.append(nxt)
    xs = np.array(xs)
    ys = f(xs)

    factor = 1.0 - 2.0 * eta
    a = abs(factor)
    if abs(eta - 0.5) < 1e-9:
        regime, col = 'one-step landing (\u03b7 = \u00bd)', GREEN
    elif a < 1e-9:
        regime, col = 'one-step landing', GREEN
    elif factor > 0 and a < 1:
        regime, col = 'smooth glide  \u2713', GREEN
    elif factor < 0 and a < 1:
        regime, col = 'shrinking spiral  \u2713', ORANGE
    elif a == 1 or abs(a - 1) < 1e-9:
        regime, col = 'stuck bounce  \u2717', RED
    else:
        regime, col = 'explosion  \u2717', RED

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(11.0, 4.6))

    # --- left: the path on the bowl ---
    span = max(abs(x0) * 1.3, np.nanmax(np.abs(xs[np.isfinite(xs)])) * 1.05, 1.0)
    span = min(span, 6.0)
    gx = np.linspace(-span, span, 400)
    axes(axL, 'descent path on the bowl  f(x) = x\u00b2')
    axL.plot(gx, f(gx), color=BLUE, lw=2.4, zorder=1, label='f(x) = x\u00b2')
    vis = np.isfinite(xs) & (np.abs(xs) <= span)
    axL.plot(xs[vis], ys[vis], '-', color=RED, lw=1.6, zorder=3)
    axL.scatter(xs[vis], ys[vis], s=42, color=RED, zorder=4,
                label='descent steps')
    axL.scatter([x0], [f(x0)], s=95, color=DARK, zorder=5,
                marker='s', label='start  x\u2080')
    # Mark the converged point green if we actually settled near 0.
    if np.isfinite(xs[-1]) and abs(xs[-1]) < 1e-2:
        axL.scatter([xs[-1]], [f(xs[-1])], s=150, color=GREEN,
                    zorder=6, marker='*', label='converged \u2192 0')
    axL.set_xlabel('x'); axL.set_ylabel('f(x)')
    axL.set_xlim(-span, span); axL.set_ylim(-0.05 * span**2, span**2)
    axL.legend(loc='upper center', framealpha=0.9, fontsize=9)

    # --- right: x_k versus step number ---
    axes(axR, f'regime:  {regime}')
    k = np.arange(len(xs))
    axR.axhline(0.0, color=GRAY, lw=1.0, ls=':')
    kk = k[vis]; xx = xs[vis]
    axR.plot(kk, xx, '-o', color=col, ms=6, zorder=3)
    axR.set_xlabel('step k'); axR.set_ylabel('x_k')
    axR.set_title(f'regime:  {regime}', color=col)
    axR.text(0.5, 1.12, f'factor 1\u22122\u03b7 = {factor:+.3f}'
             f'   (|1\u22122\u03b7| = {a:.3f})',
             transform=axR.transAxes, ha='center', color=DARK,
             fontsize=11)
    if np.all(np.isfinite(xs)):
        ylo, yhi = np.min(xs), np.max(xs)
        pad = max(0.1, 0.12 * (yhi - ylo))
        axR.set_ylim(ylo - pad, yhi + pad)
    fig.tight_layout()
    plt.show()

    # A short numeric read-out of the first several step values.
    shown = ', '.join(f'{v:.4g}' for v in xs[:7])
    tail = ' \u2026' if len(xs) > 7 else ''
    display(Markdown(f'**Steps:** $x_0,\\,x_1,\\,\\ldots$ = '
                     f'`{shown}{tail}`'))

interact(_playground,
         x0=FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1,
                        description='start x\u2080'),
         eta=FloatSlider(value=0.1, min=0.01, max=1.2, step=0.01,
                         description='\u03b7 (rate)', readout_format='.2f'),
         n_steps=IntSlider(value=12, min=2, max=30, step=1,
                           description='steps'));

interactive(children=(FloatSlider(value=1.0, description='start x₀', max=3.0, min=-3.0), FloatSlider(value=0.1…

## 2. Newton's method — fast when it works, spectacular when it doesn't

Newton's method replaces the hard curve with its easy tangent line and jumps to where the *line* hits zero:

$$x_{n+1}=x_n-\frac{f(x_n)}{f'(x_n)}.$$

Pick **$g(x)=x^2-2$** (its root is $\sqrt2$) and watch the *correct digits roughly double every step* — $2,5,11,\ldots$ — from a sloppy start. That blistering speed is **quadratic convergence**, Newton's signature. Then switch to **$x^{1/3}$**, whose only root is $0$: its rule simplifies to $x_{n+1}=-2x_n$, so the guess flips sign and doubles in size every step — $1,-2,4,-8,\ldots$ — running screaming away from the very root it was meant to find. Same elegant formula; the difference is whether the slope behaves at the root.

In [3]:
def _newton_panel(which='g(x) = x\u00b2 \u2212 2   (root \u221a2)', steps=4):
    if which.startswith('g(x)'):
        f  = lambda x: x**2 - 2.0
        fp = lambda x: 2.0 * x
        x0, root = 1.5, np.sqrt(2.0)
        title = 'Newton on  g(x) = x\u00b2 \u2212 2   \u2192   \u221a2'
        gx = np.linspace(0.4, 2.2, 400)
        good = True
    else:
        # Real cube root, sign-aware: cbrt blows up under Newton.
        f  = lambda x: np.cbrt(x)
        fp = lambda x: (1.0 / 3.0) * np.cbrt(x) / np.where(x == 0, np.nan, x)
        x0, root = 1.0, 0.0
        title = 'Newton on  f(x) = x^(1/3)   (root 0) \u2014 diverges'
        gx = np.linspace(-9.0, 9.0, 400)
        good = False

    # Run Newton, recording each guess.
    xs = [float(x0)]
    for _ in range(steps):
        x = xs[-1]
        d = fp(x)
        if d == 0 or not np.isfinite(d):
            break
        nxt = x - f(x) / d
        if not np.isfinite(nxt) or abs(nxt) > 1e9:
            xs.append(nxt); break
        xs.append(nxt)
    xs = np.array(xs)

    fig, ax = plt.subplots(figsize=(7.6, 4.6))
    axes(ax, title)
    ax.axhline(0.0, color=GRAY, lw=1.0)
    ax.plot(gx, f(gx), color=BLUE, lw=2.4, zorder=1)
    ax.scatter([root], [0.0], s=110, color=GREEN, zorder=6,
               marker='*', label=f'true root = {root:.4g}')
    # Draw the tangent slides for the first few steps (the geometry).
    shown = min(len(xs) - 1, 4)
    for i in range(shown):
        x = xs[i]
        if not np.isfinite(x):
            continue
        ax.plot([x, x], [0.0, f(x)], color=PURPLE, lw=1.0, ls=':',
                zorder=2)
        ax.plot([x, xs[i + 1]], [f(x), 0.0], color=PURPLE, lw=1.4,
                zorder=2)
    vis = np.isfinite(xs) & (np.abs(xs) <= np.abs(gx).max())
    ax.scatter(xs[vis], f(xs[vis]), s=45, color=PURPLE, zorder=5,
               label='Newton guesses')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_xlim(gx.min(), gx.max())
    ax.legend(loc='upper left', framealpha=0.9, fontsize=9)
    plt.show()

    # The convergence table — the heart of this panel.
    rows = ['| step | guess $x_n$ | error $|x_n-\\text{root}|$ | correct digits |',
            '|:---:|:---|:---|:---:|']
    for n, x in enumerate(xs):
        if not np.isfinite(x):
            rows.append(f'| $x_{{{n}}}$ | overflow | \u2014 | \u2014 |')
            continue
        err = abs(x - root)
        if err == 0:
            dig = '\u221e'
        elif err >= 1:
            dig = '0'
        else:
            dig = f'{-np.log10(err):.1f}'
        rows.append(f'| $x_{{{n}}}$ | {x:.12g} | {err:.3e} | {dig} |')
    note = ('Watch the **correct digits roughly double** each step '
            '\u2014 quadratic convergence.' if good else
            'The error **doubles** each step ($x_{n+1}=-2x_n$): '
            'Newton runs away from the root.')
    display(Markdown('\n'.join(rows) + '\n\n' + note))

interact(_newton_panel,
         which=Dropdown(
             options=['g(x) = x\u00b2 \u2212 2   (root \u221a2)',
                      'f(x) = x^(1/3)   (root 0, blows up)'],
             value='g(x) = x\u00b2 \u2212 2   (root \u221a2)',
             description='function'),
         steps=IntSlider(value=4, min=1, max=10, step=1,
                         description='steps'));

interactive(children=(Dropdown(description='function', options=('g(x) = x² − 2   (root √2)', 'f(x) = x^(1/3)  …

## 3. Two valleys — where you *start* decides where you *land*

Real loss landscapes are not single bowls; they have several valleys, and gradient descent only ever finds *a* nearby one — not necessarily the best one. Here is the smallest landscape that shows it, $f(x)=x^4-3x^2+x$. Its slope is $f'(x)=4x^3-6x+1$, which is zero at three places: two **minima** (a shallow one on the left, a deeper one on the right) with a hill between them.

Drag the **start** $x_0$ and watch the red path roll downhill. Start on the left of the hill and you fall into the shallow (worse) minimum; start on the right and you reach the deeper (better) one. Nothing about the descent changed — only where you began. *That* is the difference between a **local** and a **global** minimum, and it is the central headache of training a model: gradient descent finds the valley you happened to be standing over.

In [4]:
# Precompute the two true minima of f(x)=x^4-3x^2+x via f'(x)=4x^3-6x+1.
_F3   = lambda x: x**4 - 3.0 * x**2 + x
_F3P  = lambda x: 4.0 * x**3 - 6.0 * x + 1.0
_CRIT = np.roots([4.0, 0.0, -6.0, 1.0]).real
_CRIT = np.sort(_CRIT)
_MINS = np.array([c for c in _CRIT if (12.0 * c**2 - 6.0) > 0])  # f''>0

def _two_valleys(x0=-1.6, eta=0.02, n_steps=40):
    xs = [float(x0)]
    for _ in range(n_steps):
        nxt = xs[-1] - eta * _F3P(xs[-1])
        if not np.isfinite(nxt) or abs(nxt) > 1e6:
            xs.append(nxt); break
        xs.append(nxt)
        if abs(_F3P(xs[-1])) < 1e-9:
            break
    xs = np.array(xs)
    landed = xs[-1]
    # Which true minimum did we land nearest?
    target = _MINS[np.argmin(np.abs(_MINS - landed))] if np.isfinite(landed) else np.nan
    glob = _MINS[np.argmin(_F3(_MINS))]
    is_global = np.isfinite(landed) and abs(target - glob) < 1e-6

    fig, ax = plt.subplots(figsize=(8.2, 4.8))
    axes(ax, 'two-valley landscape  f(x) = x\u2074 \u2212 3x\u00b2 + x')
    gx = np.linspace(-2.1, 2.1, 500)
    ax.plot(gx, _F3(gx), color=BLUE, lw=2.4, zorder=1, label='f(x)')
    # Mark both minima; star the global one.
    for m in _MINS:
        is_g = abs(m - glob) < 1e-6
        ax.scatter([m], [_F3(m)], s=150 if is_g else 90,
                   color=GREEN, zorder=5,
                   marker='*' if is_g else 'o',
                   label=('global min' if is_g else 'local min'))
    vis = np.isfinite(xs)
    ax.plot(xs[vis], _F3(xs[vis]), '-', color=RED, lw=1.4, zorder=3)
    ax.scatter(xs[vis], _F3(xs[vis]), s=26, color=RED, zorder=4,
               label='descent path')
    ax.scatter([x0], [_F3(x0)], s=110, color=DARK, marker='s',
               zorder=6, label='start x\u2080')
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    ax.set_xlim(-2.1, 2.1)
    ax.set_ylim(_F3(gx).min() - 0.4, _F3(_CRIT[1]) + 1.0)
    ax.legend(loc='upper center', framealpha=0.9, fontsize=9, ncol=2)
    plt.show()

    verdict = ('reached the **global** minimum \u2713' if is_global
               else 'trapped in a **local** (worse) minimum \u2717')
    display(Markdown(
        f'Started at $x_0={x0:.2f}$ \u2192 landed at '
        f'$x\\approx{landed:.4f}$ with '
        f'$f={_F3(landed):.4f}$ \u2014 {verdict}.  '
        f'(Global min sits at $x\\approx{glob:.4f}$, '
        f'$f\\approx{_F3(glob):.4f}$.)'))

interact(_two_valleys,
         x0=FloatSlider(value=-1.6, min=-2.0, max=2.0, step=0.05,
                        description='start x\u2080'),
         eta=FloatSlider(value=0.02, min=0.005, max=0.06, step=0.005,
                         description='\u03b7 (rate)', readout_format='.3f'),
         n_steps=IntSlider(value=40, min=5, max=80, step=5,
                           description='steps'));

interactive(children=(FloatSlider(value=-1.6, description='start x₀', max=2.0, min=-2.0, step=0.05), FloatSlid…

## 4. The race — Newton vs. gradient descent, counted in steps

The two methods are secretly the same idea: both step against the slope $f'$. Gradient descent uses a fixed step size $\eta$ that *you* pick; Newton uses $1/f''(x_n)$, reading the curvature off the function and setting its own step size. Let us race them on the same job — minimizing a convex $f$ by driving $f'$ to zero — and **count the steps each needs** to get within a tolerance.

On a quadratic $f(x)=\tfrac{a}{2}x^2$ Newton lands in a **single step** from anywhere (its automatic step size $1/f''=1/a$ is exactly the perfect $\eta$). Gradient descent needs many, and how many depends on the $\eta$ you chose. Slide $\eta$ and the curvature $a$ and watch Newton win by a landslide — the lesson behind *why* Newton is the fast method (and Section 5 shows the catch that keeps AI from using it).

In [5]:
def _race(a=2.0, eta=0.3, x0=2.5, tol=1e-6, max_steps=200):
    f   = lambda x: 0.5 * a * x**2
    fp  = lambda x: a * x
    fpp = lambda x: a
    xmin = 0.0

    # Gradient descent on f.
    gd = [float(x0)]
    for _ in range(max_steps):
        gd.append(gd[-1] - eta * fp(gd[-1]))
        if not np.isfinite(gd[-1]) or abs(gd[-1]) > 1e9:
            break
        if abs(gd[-1] - xmin) < tol:
            break
    gd = np.array(gd)
    gd_steps = len(gd) - 1
    gd_ok = np.isfinite(gd[-1]) and abs(gd[-1] - xmin) < tol

    # Newton on f' (i.e. x - f'/f'') to find the minimum.
    nt = [float(x0)]
    for _ in range(max_steps):
        nt.append(nt[-1] - fp(nt[-1]) / fpp(nt[-1]))
        if abs(nt[-1] - xmin) < tol:
            break
    nt = np.array(nt)
    nt_steps = len(nt) - 1

    fig, ax = plt.subplots(figsize=(8.0, 4.6))
    axes(ax, f'distance to the minimum vs. step   (a = {a:.2g}, \u03b7 = {eta:.2g})')
    ax.semilogy(np.arange(len(gd)), np.abs(gd - xmin) + 1e-16, '-o',
                color=RED, ms=5, label=f'gradient descent ({gd_steps} steps)')
    ax.semilogy(np.arange(len(nt)), np.abs(nt - xmin) + 1e-16, '-s',
                color=PURPLE, ms=6, label=f'Newton ({nt_steps} step'
                f'{"s" if nt_steps != 1 else ""})')
    ax.axhline(tol, color=GREEN, lw=1.4, ls='--', label=f'tolerance {tol:g}')
    ax.set_xlabel('step'); ax.set_ylabel('|x \u2212 x*|  (log scale)')
    ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
    plt.show()

    winner = ('Newton' if nt_steps < gd_steps else
              'gradient descent' if gd_steps < nt_steps else 'a tie')
    gd_txt = f'**{gd_steps}** steps' + ('' if gd_ok else ' (did not converge)')
    display(Markdown(
        f'**Steps to reach tolerance {tol:g}:**  '
        f'Newton = **{nt_steps}**, gradient descent = {gd_txt}.  '
        f'Winner: **{winner}**.  '
        f'(On a quadratic, Newton\u2019s step size $1/f\'\'=1/{a:.2g}$ '
        f'is the exact best $\\eta$, so it lands in one step.)'))

interact(_race,
         a=FloatSlider(value=2.0, min=0.5, max=8.0, step=0.5,
                       description='curvature a'),
         eta=FloatSlider(value=0.3, min=0.02, max=0.9, step=0.02,
                         description='\u03b7 (GD rate)', readout_format='.2f'),
         x0=FloatSlider(value=2.5, min=-4.0, max=4.0, step=0.5,
                        description='start x\u2080'));

interactive(children=(FloatSlider(value=2.0, description='curvature a', max=8.0, min=0.5, step=0.5), FloatSlid…

## 5. Your first real optimizer — momentum on a steep bowl

On a *steep*, ill-conditioned bowl $f(x)=k\,x^2$ (large $k$ means sharp curvature), plain gradient descent is forced to take tiny steps: its safe range $0<\eta<\tfrac1k$ collapses as $k$ grows, so it crawls. **Momentum** — the heavy-ball method — fixes this by letting the steps build up speed, like a ball gathering momentum rolling downhill:

$$v_{k+1}=\beta\,v_k-\eta\,f'(x_k),\qquad x_{k+1}=x_k+v_{k+1}.$$

The velocity $v$ remembers past steps (the factor $\beta$, here $0.9$); plain gradient descent is just the $\beta=0$ case. Tick the **momentum** box to overlay the heavy-ball path (teal) on the plain one (red), and compare how many steps each needs. This is a baby version of the optimizers that actually train neural networks — and your bridge to Book 3, where $x$ becomes a vector and this exact rule does the heavy lifting.

In [6]:
def _momentum(k=10.0, eta=0.05, beta=0.9, use_momentum=True,
              x0=1.0, tol=1e-4, max_steps=120):
    fp = lambda x: 2.0 * k * x
    f  = lambda x: k * x**2
    xmin = 0.0

    def run(mom):
        xs = [float(x0)]; v = 0.0
        for _ in range(max_steps):
            v = (beta * v if mom else 0.0) - eta * fp(xs[-1])
            nxt = xs[-1] + v
            xs.append(nxt)
            if not np.isfinite(nxt) or abs(nxt) > 1e9:
                break
            if abs(nxt - xmin) < tol:
                break
        return np.array(xs)

    plain = run(False)
    heavy = run(True)

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(11.0, 4.6))

    # --- left: paths on the steep bowl ---
    axes(axL, f'paths on the steep bowl  f(x) = {k:.0f}\u00b7x\u00b2')
    span = max(abs(x0) * 1.2, 1.0)
    gx = np.linspace(-span, span, 400)
    axL.plot(gx, f(gx), color=BLUE, lw=2.2, zorder=1, label='f(x)')
    pv = np.isfinite(plain) & (np.abs(plain) <= span)
    axL.plot(plain[pv], f(plain[pv]), '-o', color=RED, ms=4, zorder=3,
             label=f'plain GD ({len(plain) - 1} steps)')
    if use_momentum:
        hv = np.isfinite(heavy) & (np.abs(heavy) <= span)
        axL.plot(heavy[hv], f(heavy[hv]), '-s', color=TEAL, ms=5,
                 zorder=4, label=f'momentum ({len(heavy) - 1} steps)')
    axL.scatter([0], [0], s=140, color=GREEN, marker='*', zorder=6)
    axL.set_xlabel('x'); axL.set_ylabel('f(x)')
    axL.set_xlim(-span, span); axL.set_ylim(-0.05 * f(span), f(span))
    axL.legend(loc='upper center', framealpha=0.9, fontsize=9)

    # --- right: convergence speed ---
    axes(axR, 'distance to minimum vs. step')
    axR.semilogy(np.arange(len(plain)), np.abs(plain - xmin) + 1e-16,
                 '-o', color=RED, ms=4, label='plain GD')
    if use_momentum:
        axR.semilogy(np.arange(len(heavy)), np.abs(heavy - xmin) + 1e-16,
                     '-s', color=TEAL, ms=5, label='momentum')
    axR.axhline(tol, color=GREEN, lw=1.3, ls='--', label=f'tol {tol:g}')
    axR.set_xlabel('step'); axR.set_ylabel('|x \u2212 x*|  (log)')
    axR.legend(loc='upper right', framealpha=0.9, fontsize=9)
    fig.tight_layout()
    plt.show()

    if use_momentum:
        p, h = len(plain) - 1, len(heavy) - 1
        verdict = (f'Momentum reached tolerance in **{h}** steps vs. '
                   f'**{p}** for plain gradient descent'
                   + (f' \u2014 a {p / h:.1f}\u00d7 speed-up.' if h and h < p
                      else '.'))
    else:
        verdict = ('Momentum is **off** ($\\beta=0$): this is plain '
                   'gradient descent. Tick the box to add the heavy ball.')
    display(Markdown(verdict))

interact(_momentum,
         k=FloatSlider(value=10.0, min=1.0, max=30.0, step=1.0,
                       description='steepness k'),
         eta=FloatSlider(value=0.05, min=0.005, max=0.09, step=0.005,
                         description='\u03b7 (rate)', readout_format='.3f'),
         beta=FloatSlider(value=0.9, min=0.0, max=0.95, step=0.05,
                          description='\u03b2 (momentum)', readout_format='.2f'),
         use_momentum=Checkbox(value=True, description='momentum on'));

interactive(children=(FloatSlider(value=10.0, description='steepness k', max=30.0, min=1.0, step=1.0), FloatSl…

---

*Every widget here is the same chapter, run by hand: a guess that gets better each step. You watched a learning rate slide a glide into an explosion, Newton double its digits and then fly apart, a starting point alone decide local versus global, and a heavy ball outrun plain descent on a steep bowl. In Book 3 the single number $x$ becomes a vector and $f'$ becomes the **gradient** — but the update rule $x_{k+1}=x_k-\eta\,(\text{gradient})$ keeps the exact same shape. When you next watch a neural network's loss tick downward, you will be watching this notebook, run a few million times.*